In [80]:
# ============================================================================
# IMPORTS
# ============================================================================
import pandas as pd
import numpy as np

# Load Data 

In [81]:
# ============================================================================
# STEP 1: Load Human-Labeled Data (Train/Test Split)
# ============================================================================
# These are the video IDs that were manually labeled by humans
# and already split into training and test sets

# Load human-labeled training video IDs
human_labeled_train_video_ids = pd.read_csv('data/train_y.csv').set_index("video_id").index.to_list()

# Load human-labeled test video IDs
human_labeled_test_video_ids = pd.read_csv('data/test_y.csv').set_index("video_id").index.to_list()

# ============================================================================
# STEP 2: Load AI-Labeled Data
# ============================================================================
# Load the AI-labeled dataset (contains video_id and is_understandable labels)
ai_labeled_data = pd.read_excel('data/youtube_database_AI.xlsx', sheet_name='LabeledVideos')
# Keep only video_id and is_understandable columns, set video_id as index
ai_labeled_data = ai_labeled_data.loc[:, ["video_id", "is_understandable"]].set_index("video_id")

print(f"✅ Loaded {len(human_labeled_train_video_ids)} human-labeled training videos")
print(f"✅ Loaded {len(human_labeled_test_video_ids)} human-labeled test videos")
print(f"✅ Loaded {len(ai_labeled_data)} AI-labeled videos")

✅ Loaded 140 human-labeled training videos
✅ Loaded 50 human-labeled test videos
✅ Loaded 1163 AI-labeled videos


In [82]:
# ============================================================================
# STEP 3: Load Feature Data Files First (to check available video IDs)
# ============================================================================
# Load feature files to see which video IDs are available in the feature data
print("📂 Loading feature files...")
x_train_features_raw = pd.read_csv('data/x_train_scaled.csv').set_index("video_id")
x_test_features_raw = pd.read_csv('data/x_test_scaled.csv').set_index("video_id")

available_train_video_ids = set(x_train_features_raw.index)
available_test_video_ids = set(x_test_features_raw.index)

print(f"   - Available training video IDs in features: {len(available_train_video_ids)}")
print(f"   - Available test video IDs in features: {len(available_test_video_ids)}")

# ============================================================================
# STEP 4: Extract AI-Labeled Data for Train/Test Sets
# ============================================================================
# Find AI-labeled videos that match the human-labeled train/test splits
# AND have features available in the feature files

# Get AI-labeled video IDs that are in the human training set
ai_labeled_train_video_ids_all = ai_labeled_data[
    ai_labeled_data.index.isin(human_labeled_train_video_ids)
].index.to_list()

# Get AI-labeled video IDs that are in the human test set
ai_labeled_test_video_ids_all = ai_labeled_data[
    ai_labeled_data.index.isin(human_labeled_test_video_ids)
].index.to_list()

# Filter to only include video IDs that have features available
ai_labeled_train_video_ids = [
    vid for vid in ai_labeled_train_video_ids_all 
    if vid in available_train_video_ids
]

ai_labeled_test_video_ids = [
    vid for vid in ai_labeled_test_video_ids_all 
    if vid in available_test_video_ids
]

# Report any missing videos
missing_train_videos = set(ai_labeled_train_video_ids_all) - set(ai_labeled_train_video_ids)
missing_test_videos = set(ai_labeled_test_video_ids_all) - set(ai_labeled_test_video_ids)

if missing_train_videos:
    print(f"\n⚠️  Warning: {len(missing_train_videos)} training videos have AI labels but no features:")
    print(f"   Missing video IDs: {list(missing_train_videos)[:5]}..." if len(missing_train_videos) > 5 else f"   Missing video IDs: {list(missing_train_videos)}")

if missing_test_videos:
    print(f"\n⚠️  Warning: {len(missing_test_videos)} test videos have AI labels but no features:")
    print(f"   Missing video IDs: {list(missing_test_videos)[:5]}..." if len(missing_test_videos) > 5 else f"   Missing video IDs: {list(missing_test_videos)}")

# ============================================================================
# STEP 5: Filter Feature Data (X) to Match Available Video IDs
# ============================================================================
# Now safely filter features using only video IDs that exist in both datasets

# Filter training features to only include videos with both AI labels and features
x_train_features = x_train_features_raw.loc[ai_labeled_train_video_ids]

# Filter test features to only include videos with both AI labels and features
x_test_features = x_test_features_raw.loc[ai_labeled_test_video_ids]

# ============================================================================
# STEP 6: Extract Labels (y) for Train/Test Sets
# ============================================================================
# Get AI labels for videos that have both labels and features

# Training labels
train_labels_ai = ai_labeled_data[ai_labeled_data.index.isin(ai_labeled_train_video_ids)]

# Test labels
test_labels_ai = ai_labeled_data[ai_labeled_data.index.isin(ai_labeled_test_video_ids)]

# ============================================================================
# Summary Statistics
# ============================================================================
print("\n" + "=" * 80)
print("DATA SUMMARY")
print("=" * 80)
print(f"📊 Training set:")
print(f"   - AI-labeled training video IDs (with features): {len(ai_labeled_train_video_ids)}")
print(f"   - Training features shape: {x_train_features.shape}")
print(f"   - Training labels shape: {train_labels_ai.shape}")
print(f"\n📊 Test set:")
print(f"   - AI-labeled test video IDs (with features): {len(ai_labeled_test_video_ids)}")
print(f"   - Test features shape: {x_test_features.shape}")
print(f"   - Test labels shape: {test_labels_ai.shape}")
print("=" * 80)

📂 Loading feature files...
   - Available training video IDs in features: 133
   - Available test video IDs in features: 49

⚠️  Warning: 7 training videos have AI labels but no features:
   Missing video IDs: ['lxVGPACR5ok', 'i1Tb5gfymfU', 'aKmcTOY4N0M', '4pixAOy_yrU', 'DgBYYFwd42A']...

⚠️  Warning: 1 test videos have AI labels but no features:
   Missing video IDs: ['3aUEf7VZLas']

DATA SUMMARY
📊 Training set:
   - AI-labeled training video IDs (with features): 129
   - Training features shape: (129, 31)
   - Training labels shape: (129, 1)

📊 Test set:
   - AI-labeled test video IDs (with features): 48
   - Test features shape: (48, 31)
   - Test labels shape: (48, 1)


In [83]:
# ============================================================================
# STEP 6: Expand Training Set with Additional AI-Labeled Videos
# ============================================================================
# Strategy: Sample additional videos from unlabeled pool and add them to training set
# This increases the training data size using AI labels

# ============================================================================
# STEP 6.1: Load Candidate Videos (Unlabeled Pool)
# ============================================================================
# These are videos that haven't been used in train/test split yet
# but have AI labels available
candidate_video_ids = pd.read_csv('data/to_predict_x.csv').set_index("video_id").index.to_list()
print(f"📋 Total candidate videos available: {len(candidate_video_ids)}")

# ============================================================================
# STEP 6.2: Randomly Sample Videos from Candidate Pool
# ============================================================================
# Sample a subset of videos to add to training set
np.random.seed(42)  # For reproducibility
n_samples_to_add = 900  # Number of videos to sample
sampled_candidate_video_ids = np.random.choice(
    candidate_video_ids, 
    size=n_samples_to_add, 
    replace=False  # No duplicates
)
print(f"✅ Randomly sampled {len(sampled_candidate_video_ids)} videos from candidate pool")

# ============================================================================
# STEP 6.3: Filter to Only Videos with Both AI Labels AND Features
# ============================================================================
# Only keep sampled videos that have:
# 1. AI labels in our dataset
# 2. Features available in to_predict_x.csv

# First, check which candidate videos have features
candidate_features = pd.read_csv('data/to_predict_x.csv').set_index("video_id")
available_candidate_video_ids = set(candidate_features.index)

# Filter sampled videos to only those with AI labels
sampled_videos_with_ai_labels = ai_labeled_data[
    ai_labeled_data.index.isin(sampled_candidate_video_ids)
].index.to_list()

# Further filter to only those with features available
new_train_video_ids_from_sampling = [
    vid for vid in sampled_videos_with_ai_labels 
    if vid in available_candidate_video_ids
]

print(f"✅ Found {len(sampled_videos_with_ai_labels)} sampled videos with AI labels")
print(f"✅ Found {len(new_train_video_ids_from_sampling)} sampled videos with both AI labels and features")

# Report any missing features
missing_features = set(sampled_videos_with_ai_labels) - set(new_train_video_ids_from_sampling)
if missing_features:
    print(f"⚠️  Warning: {len(missing_features)} videos have AI labels but no features (will be skipped)")

# ============================================================================
# STEP 6.4: Merge New Videos into Training Set
# ============================================================================
# Add the newly sampled videos to the existing training video IDs
# Note: Using extend() to merge lists (not append() which adds a single element)
ai_labeled_train_video_ids.extend(new_train_video_ids_from_sampling)

# ============================================================================
# STEP 6.5: Update Training Labels (y)
# ============================================================================
# Get all AI labels for the expanded training set
final_train_labels_ai = ai_labeled_data[
    ai_labeled_data.index.isin(ai_labeled_train_video_ids)
]

# ============================================================================
# STEP 6.6: Update Training Features (X)
# ============================================================================
# Extract features for the newly sampled videos (already loaded in step 6.3)
x_train_new_samples = candidate_features.loc[new_train_video_ids_from_sampling]

# Concatenate original training features with new samples
x_train_features = pd.concat([x_train_features, x_train_new_samples])

# ============================================================================
# Final Summary
# ============================================================================
print("\n" + "=" * 80)
print("EXPANDED TRAINING SET SUMMARY")
print("=" * 80)
print(f"📊 Original training set:")
print(f"   - Video count: {len(ai_labeled_train_video_ids) - len(new_train_video_ids_from_sampling)}")
print(f"\n📊 Newly added videos:")
print(f"   - Video count: {len(new_train_video_ids_from_sampling)}")
print(f"\n📊 Final expanded training set:")
print(f"   - Total video count: {len(ai_labeled_train_video_ids)}")
print(f"   - Training labels shape: {final_train_labels_ai.shape}")
print(f"   - Training features shape: {x_train_features.shape}")
print("=" * 80)


📋 Total candidate videos available: 2297
✅ Randomly sampled 900 videos from candidate pool
✅ Found 376 sampled videos with AI labels
✅ Found 376 sampled videos with both AI labels and features

EXPANDED TRAINING SET SUMMARY
📊 Original training set:
   - Video count: 129

📊 Newly added videos:
   - Video count: 376

📊 Final expanded training set:
   - Total video count: 505
   - Training labels shape: (505, 1)
   - Training features shape: (505, 31)


# Modelling


In [84]:
x_train = x_train_features.sort_index()
y_train = final_train_labels_ai.sort_index()


x_test = x_test_features.sort_index()
y_test = test_labels_ai.sort_index()


In [94]:
# ============================================================================
# DATA ALIGNMENT VERIFICATION
# ============================================================================
# Verify that X (features) and y (labels) are properly aligned by video_id
# This is critical for model training - mismatched indices will cause errors

print("=" * 80)
print("VERIFYING DATA ALIGNMENT")
print("=" * 80)

# ============================================================================
# Training Set Alignment Check
# ============================================================================
print("\n📊 Training Set Alignment:")
print(f"   x_train shape: {x_train.shape}")
print(f"   y_train shape: {y_train.shape}")

# Check if indices match
train_indices_match = x_train.index.equals(y_train.index)
print(f"   ✅ Indices match: {train_indices_match}")

if not train_indices_match:
    print("\n   ⚠️  WARNING: Training set indices do not match!")
    # Find differences
    train_x_only = set(x_train.index) - set(y_train.index)
    train_y_only = set(y_train.index) - set(x_train.index)
    
    if train_x_only:
        print(f"   - Videos in x_train but not in y_train: {len(train_x_only)}")
        print(f"     Examples: {list(train_x_only)[:3]}")
    if train_y_only:
        print(f"   - Videos in y_train but not in x_train: {len(train_y_only)}")
        print(f"     Examples: {list(train_y_only)[:3]}")
else:
    print("   ✅ All training video IDs are aligned correctly!")

# ============================================================================
# Test Set Alignment Check
# ============================================================================
print("\n📊 Test Set Alignment:")
print(f"   x_test shape: {x_test.shape}")
print(f"   y_test shape: {y_test.shape}")

# Check if indices match
test_indices_match = x_test.index.equals(y_test.index)
print(f"   ✅ Indices match: {test_indices_match}")

if not test_indices_match:
    print("\n   ⚠️  WARNING: Test set indices do not match!")
    # Find differences
    test_x_only = set(x_test.index) - set(y_test.index)
    test_y_only = set(y_test.index) - set(x_test.index)
    
    if test_x_only:
        print(f"   - Videos in x_test but not in y_test: {len(test_x_only)}")
        print(f"     Examples: {list(test_x_only)[:3]}")
    if test_y_only:
        print(f"   - Videos in y_test but not in x_test: {len(test_y_only)}")
        print(f"     Examples: {list(test_y_only)[:3]}")
else:
    print("   ✅ All test video IDs are aligned correctly!")

# ============================================================================
# Detailed Comparison DataFrame (for inspection if needed)
# ============================================================================
# Create comparison DataFrames for detailed inspection
train_comparison = pd.DataFrame({
    'x_train_index': x_train.index,
    'y_train_index': y_train.index
})
train_comparison['indices_match'] = train_comparison['x_train_index'] == train_comparison['y_train_index']

test_comparison = pd.DataFrame({
    'x_test_index': x_test.index,
    'y_test_index': y_test.index
})
test_comparison['indices_match'] = test_comparison['x_test_index'] == test_comparison['y_test_index']

print("\n" + "=" * 80)
print("DETAILED ALIGNMENT SUMMARY")
print("=" * 80)
print(f"\n📊 Training Set:")
print(f"   Total videos: {len(train_comparison)}")
print(f"   Matching indices: {train_comparison['indices_match'].sum()}")
print(f"   Mismatched indices: {(~train_comparison['indices_match']).sum()}")

print(f"\n📊 Test Set:")
print(f"   Total videos: {len(test_comparison)}")
print(f"   Matching indices: {test_comparison['indices_match'].sum()}")
print(f"   Mismatched indices: {(~test_comparison['indices_match']).sum()}")

# Show mismatches if any
if not train_comparison['indices_match'].all():
    print("\n⚠️  Training Set Mismatches:")
    print(train_comparison[~train_comparison['indices_match']].head(10))

if not test_comparison['indices_match'].all():
    print("\n⚠️  Test Set Mismatches:")
    print(test_comparison[~test_comparison['indices_match']].head(10))

print("\n" + "=" * 80)
if train_indices_match and test_indices_match:
    print("✅ ALL DATA IS PROPERLY ALIGNED - Ready for model training!")
else:
    print("⚠️  DATA MISALIGNMENT DETECTED - Please fix before training!")
print("=" * 80)

VERIFYING DATA ALIGNMENT

📊 Training Set Alignment:
   x_train shape: (505, 31)
   y_train shape: (505, 1)
   ✅ Indices match: True
   ✅ All training video IDs are aligned correctly!

📊 Test Set Alignment:
   x_test shape: (48, 31)
   y_test shape: (48, 1)
   ✅ Indices match: True
   ✅ All test video IDs are aligned correctly!

DETAILED ALIGNMENT SUMMARY

📊 Training Set:
   Total videos: 505
   Matching indices: 505
   Mismatched indices: 0

📊 Test Set:
   Total videos: 48
   Matching indices: 48
   Mismatched indices: 0

✅ ALL DATA IS PROPERLY ALIGNED - Ready for model training!


## Logistic Regression

In [105]:
# ============================================================================
# LOGISTIC REGRESSION WITH GRID SEARCH AND REGULARIZATION
# ============================================================================
# This cell implements Logistic Regression with:
# - GridSearchCV for hyperparameter tuning
# - L1 and L2 regularization
# - Cross-validation
# - Retraining on full training set with best parameters
# - Final evaluation on test set
# ============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("LOGISTIC REGRESSION: HYPERPARAMETER TUNING WITH REGULARIZATION")
print("=" * 80)

# ============================================================================
# Prepare Data for Training
# ============================================================================
# Convert y_train and y_test to numpy arrays (extract the label column)
# Assuming the label column is 'is_understandable'

# Get the label column name
label_col = y_train.columns[0] if len(y_train.columns) == 1 else 'is_understandable'

# Extract features and labels as numpy arrays
X_train = x_train.values  # Convert DataFrame to numpy array
X_test = x_test.values
y_train_values = y_train[label_col].values  # Extract label values
y_test_values = y_test[label_col].values

print(f"\n📊 Data Preparation:")
print(f"   Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"   Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"   Label column: {label_col}")
print(f"   Training labels distribution:")
print(f"      - Class 0: {np.sum(y_train_values == 0)} ({np.sum(y_train_values == 0)/len(y_train_values)*100:.1f}%)")
print(f"      - Class 1: {np.sum(y_train_values == 1)} ({np.sum(y_train_values == 1)/len(y_train_values)*100:.1f}%)")

# ============================================================================
# Define Parameter Grid for GridSearchCV
# ============================================================================
# Explore different regularization types and strengths
param_grid = {
    # Regularization type: 'l1' (Lasso) or 'l2' (Ridge)
    'penalty': ['l1', 'l2'],
    
    # Regularization strength (C is inverse of regularization strength)
    # Smaller C = stronger regularization (prevents overfitting)
    # Larger C = weaker regularization (may overfit)
    'C': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0],
    
    # Solver: different algorithms for optimization
    # 'liblinear' works with both L1 and L2
    # 'lbfgs', 'newton-cg', 'sag', 'saga' work with L2 only
    'solver': ['liblinear', 'lbfgs', 'saga'],
    
    # Maximum number of iterations
    'max_iter': [100, 200, 500, 1000],
    
    # Class weight: handle class imbalance
    # Format: {0: weight_for_class_0, 1: weight_for_class_1}
    # Higher weight for class 0 (minority class) to improve recall
    'class_weight': [
        {0: 3, 1: 1},   # 3:1 ratio (Class 0 gets 3x weight)
        {0: 4, 1: 1},   # 4:1 ratio
        {0: 5, 1: 1},   # 5:1 ratio
        {0: 6, 1: 1},   # 6:1 ratio
        {0: 7, 1: 1},   # 7:1 ratio
        {0: 8, 1: 1},   # 8:1 ratio
        {0: 9, 1: 1},   # 9:1 ratio
        {0: 10, 1: 1},  # 10:1 ratio
        'balanced'      # Automatically balanced based on class frequency
    ]
}

# Note: Some solver/penalty combinations are incompatible
# We'll handle this in the grid search

# Calculate total combinations (approximate, some will be skipped)
total_combinations = (
    len(param_grid['penalty']) * 
    len(param_grid['C']) * 
    len(param_grid['solver']) * 
    len(param_grid['max_iter']) * 
    len(param_grid['class_weight'])
)

print(f"\n🔍 Hyperparameter Grid:")
print(f"   Penalty types: {len(param_grid['penalty'])} (L1, L2)")
print(f"   C values: {len(param_grid['C'])} (regularization strength)")
print(f"   Solvers: {len(param_grid['solver'])}")
print(f"   Max iterations: {len(param_grid['max_iter'])}")
print(f"   Class weights: {len(param_grid['class_weight'])}")
print(f"   Approximate total combinations: {total_combinations}")
print(f"   With 5-fold CV: ~{total_combinations * 5} model fits")

# ============================================================================
# Setup Cross-Validation
# ============================================================================
# Use StratifiedKFold to maintain class distribution in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================================
# Initialize Logistic Regression Model
# ============================================================================
# Base model (parameters will be tuned by GridSearchCV)
base_model = LogisticRegression(random_state=42)

# ============================================================================
# Perform Grid Search with Cross-Validation
# ============================================================================
print("\n" + "=" * 80)
print("STARTING GRID SEARCH WITH CROSS-VALIDATION")
print("=" * 80)
print("\n⏳ This may take a while. Please wait...\n")

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',  # Primary metric
    n_jobs=-1,  # Use all available CPUs
    verbose=1,  # Show progress
    return_train_score=True  # Return training scores for overfitting check
)

# Fit grid search
import time
start_time = time.time()
grid_search.fit(X_train, y_train_values)
elapsed_time = time.time() - start_time

print(f"\n✅ Grid search completed!")
print(f"   Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.1f} seconds)")

# ============================================================================
# Display Best Parameters
# ============================================================================
print("\n" + "=" * 80)
print("BEST PARAMETERS FROM GRID SEARCH")
print("=" * 80)
print(f"\n🏆 Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Best Cross-Validation Score (Accuracy): {grid_search.best_score_:.4f}")

# Check for overfitting (compare train vs test CV scores)
if 'mean_train_score' in grid_search.cv_results_:
    best_idx = grid_search.best_index_
    train_cv_score = grid_search.cv_results_['mean_train_score'][best_idx]
    test_cv_score = grid_search.cv_results_['mean_test_score'][best_idx]
    cv_gap = train_cv_score - test_cv_score
    
    print(f"\n📊 Cross-Validation Scores:")
    print(f"   Train CV Score: {train_cv_score:.4f}")
    print(f"   Test CV Score: {test_cv_score:.4f}")
    print(f"   Gap: {cv_gap:.4f}")
    
    if cv_gap > 0.15:
        print(f"   ⚠️  WARNING: Large gap detected! Model may be overfitting.")
    elif cv_gap > 0.10:
        print(f"   ⚠️  CAUTION: Moderate gap detected.")
    else:
        print(f"   ✅ Good: Gap is acceptable.")

# ============================================================================
# Retrain on Full Training Set with Best Parameters
# ============================================================================
print("\n" + "=" * 80)
print("RETRAINING ON FULL TRAINING SET WITH BEST PARAMETERS")
print("=" * 80)

# Create model with best parameters
best_model = LogisticRegression(
    penalty=grid_search.best_params_['penalty'],
    C=grid_search.best_params_['C'],
    solver=grid_search.best_params_['solver'],
    max_iter=grid_search.best_params_['max_iter'],
    class_weight=grid_search.best_params_['class_weight'],
    random_state=42
)

# Retrain on FULL training set
print("\n🔄 Training on full training set...")
best_model.fit(X_train, y_train_values)
print("✅ Model trained on full training set!")

# ============================================================================
# Evaluate on Test Set
# ============================================================================
print("\n" + "=" * 80)
print("EVALUATING ON TEST SET")
print("=" * 80)

# Make predictions
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Get prediction probabilities (for ROC-AUC)
y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_test_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
train_accuracy = accuracy_score(y_train_values, y_train_pred)
test_accuracy = accuracy_score(y_test_values, y_test_pred)

train_precision = precision_score(y_train_values, y_train_pred, average='weighted', zero_division=0)
test_precision = precision_score(y_test_values, y_test_pred, average='weighted', zero_division=0)

train_recall = recall_score(y_train_values, y_train_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test_values, y_test_pred, average='weighted', zero_division=0)

train_f1 = f1_score(y_train_values, y_train_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test_values, y_test_pred, average='weighted', zero_division=0)

train_auc = roc_auc_score(y_train_values, y_train_proba)
test_auc = roc_auc_score(y_test_values, y_test_proba)

# Confusion matrices
train_cm = confusion_matrix(y_train_values, y_train_pred)
test_cm = confusion_matrix(y_test_values, y_test_pred)

# ============================================================================
# Display Results
# ============================================================================
print("\n📊 Training Set Performance:")
print(f"   Accuracy:  {train_accuracy:.4f}")
print(f"   Precision: {train_precision:.4f}")
print(f"   Recall:    {train_recall:.4f}")
print(f"   F1 Score:  {train_f1:.4f}")
print(f"   AUC-ROC:   {train_auc:.4f}")

print("\n📊 Test Set Performance (FINAL EVALUATION):")
print(f"   ✅ Test Accuracy:  {test_accuracy:.4f}")
print(f"   ✅ Test Precision: {test_precision:.4f}")
print(f"   ✅ Test Recall:    {test_recall:.4f}")
print(f"   ✅ Test F1 Score:  {test_f1:.4f}")
print(f"   ✅ Test AUC-ROC:    {test_auc:.4f}")

print("\n📊 Confusion Matrix (Test Set):")
print(test_cm)

print("\n📊 Classification Report (Test Set):")
print(classification_report(y_test_values, y_test_pred, target_names=['Not Understandable', 'Understandable']))

# Overfitting check
overfitting_gap = train_accuracy - test_accuracy
print(f"\n⚠️  Overfitting Check:")
print(f"   Train-Test Accuracy Gap: {overfitting_gap:.4f}")
if overfitting_gap > 0.15:
    print(f"   ⚠️  WARNING: Large gap detected! Model may be overfitting.")
    print(f"   💡 Consider increasing regularization strength (decreasing C).")
elif overfitting_gap > 0.10:
    print(f"   ⚠️  CAUTION: Moderate gap detected.")
    print(f"   💡 Consider slightly increasing regularization if gap increases.")
else:
    print(f"   ✅ Good: Gap is acceptable.")
    print(f"   💡 Current regularization (C={grid_search.best_params_['C']}) provides good balance.")

print("\n" + "=" * 80)
print("✅ LOGISTIC REGRESSION TRAINING COMPLETE!")
print("=" * 80)

LOGISTIC REGRESSION: HYPERPARAMETER TUNING WITH REGULARIZATION

📊 Data Preparation:
   Training set: 505 samples, 31 features
   Test set: 48 samples, 31 features
   Label column: is_understandable
   Training labels distribution:
      - Class 0: 127 (25.1%)
      - Class 1: 378 (74.9%)

🔍 Hyperparameter Grid:
   Penalty types: 2 (L1, L2)
   C values: 10 (regularization strength)
   Solvers: 3
   Max iterations: 4
   Class weights: 9
   Approximate total combinations: 2160
   With 5-fold CV: ~10800 model fits

STARTING GRID SEARCH WITH CROSS-VALIDATION

⏳ This may take a while. Please wait...

Fitting 5 folds for each of 2160 candidates, totalling 10800 fits

✅ Grid search completed!
   Total time: 0.5 minutes (29.5 seconds)

BEST PARAMETERS FROM GRID SEARCH

🏆 Best Parameters:
   C: 0.1
   class_weight: {0: 3, 1: 1}
   max_iter: 100
   penalty: l1
   solver: liblinear

📊 Best Cross-Validation Score (Accuracy): 0.7743

📊 Cross-Validation Scores:
   Train CV Score: 0.7733
   Test CV Sc

## Decision Tree

In [ ]:
# ============================================================================
# DECISION TREE: HYPERPARAMETER TUNING WITH OVERFITTING PREVENTION
# ============================================================================
# This cell implements Decision Tree with:
# - GridSearchCV for hyperparameter tuning
# - Focus on preventing overfitting (max_depth, min_samples_split, etc.)
# - Cross-validation
# - Class weight handling for imbalanced data
# - Retraining on full training set with best parameters
# - Final evaluation on test set
# ============================================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("DECISION TREE: HYPERPARAMETER TUNING WITH OVERFITTING PREVENTION")
print("=" * 80)

# ============================================================================
# Prepare Data for Training
# ============================================================================
# Reuse the same data preparation from Logistic Regression
# X_train, X_test, y_train_values, y_test_values should already be defined

print(f"\n📊 Data Preparation:")
print(f"   Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"   Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"   Training labels distribution:")
print(f"      - Class 0: {np.sum(y_train_values == 0)} ({np.sum(y_train_values == 0)/len(y_train_values)*100:.1f}%)")
print(f"      - Class 1: {np.sum(y_train_values == 1)} ({np.sum(y_train_values == 1)/len(y_train_values)*100:.1f}%)")

# ============================================================================
# Define Parameter Grid for GridSearchCV (Optimized for Accuracy)
# ============================================================================
# GridSearchCV will test ALL combinations in the parameter grid
# We balance parameter range with computation time
# Focus on parameters that balance accuracy and prevent overfitting

# Optimized parameter grid for comprehensive search
param_grid_dt = {
    # Maximum depth of the tree (CRITICAL for preventing overfitting)
    # Balanced range: not too many options to keep computation manageable
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    
    # Minimum number of samples required to split an internal node
    # Key values for preventing overfitting
    'min_samples_split': [2, 5, 10, 20, 30, 50],
    
    # Minimum number of samples required to be at a leaf node
    # Important for controlling tree complexity
    'min_samples_leaf': [1, 2, 5, 10, 20],
    
    # Maximum number of features to consider when looking for the best split
    # Common choices plus some proportion options
    'max_features': ['sqrt', 'log2', None, 0.5, 0.7],
    
    # Criterion for measuring split quality
    'criterion': ['gini', 'entropy'],
    
    # Minimum impurity decrease required for a split
    # This helps prevent overfitting by requiring meaningful splits
    'min_impurity_decrease': [0.0, 0.0001, 0.001, 0.005, 0.01],
    
    # Class weight: handle class imbalance
    # Balanced selection of weight options
    'class_weight': [
        None,  # No weighting (may work if classes are balanced enough)
        'balanced',  # Automatically balanced
        {0: 2, 1: 1},   # 2:1 ratio
        {0: 3, 1: 1},   # 3:1 ratio
        {0: 4, 1: 1},   # 4:1 ratio
        {0: 5, 1: 1},   # 5:1 ratio
        {0: 6, 1: 1},   # 6:1 ratio
    ]
}

# Calculate total combinations
total_combinations_dt = (
    len(param_grid_dt['max_depth']) * 
    len(param_grid_dt['min_samples_split']) * 
    len(param_grid_dt['min_samples_leaf']) * 
    len(param_grid_dt['max_features']) * 
    len(param_grid_dt['criterion']) * 
    len(param_grid_dt['min_impurity_decrease']) *
    len(param_grid_dt['class_weight'])
)

print(f"\n🔍 Hyperparameter Grid (GridSearchCV):")
print(f"   Max depth: {len(param_grid_dt['max_depth'])} options")
print(f"   Min samples split: {len(param_grid_dt['min_samples_split'])} options")
print(f"   Min samples leaf: {len(param_grid_dt['min_samples_leaf'])} options")
print(f"   Max features: {len(param_grid_dt['max_features'])} options")
print(f"   Criterion: {len(param_grid_dt['criterion'])} options")
print(f"   Min impurity decrease: {len(param_grid_dt['min_impurity_decrease'])} options")
print(f"   Class weights: {len(param_grid_dt['class_weight'])} options")
print(f"   Total combinations: {total_combinations_dt:,}")
print(f"   With 5-fold CV: ~{total_combinations_dt * 5:,} model fits")
print(f"\n⏰ Estimated time: ~{total_combinations_dt * 5 * 0.01:.1f} seconds (assuming ~0.01s per fit)")
print(f"💡 GridSearchCV will test ALL {total_combinations_dt:,} combinations systematically")

# ============================================================================
# Setup Cross-Validation
# ============================================================================
# Use StratifiedKFold to maintain class distribution in each fold
cv_dt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================================
# Initialize Decision Tree Model
# ============================================================================
# Base model (parameters will be tuned by GridSearchCV)
base_model_dt = DecisionTreeClassifier(random_state=42)

# ============================================================================
# Perform Grid Search with Cross-Validation
# ============================================================================
print("\n" + "=" * 80)
print("STARTING GRID SEARCH WITH CROSS-VALIDATION")
print("=" * 80)
print("\n💡 Strategy: GridSearchCV will systematically test ALL parameter")
print("   combinations in the grid to find the optimal configuration.")
print("\n⏳ This may take a while. Please wait...\n")

grid_search_dt = GridSearchCV(
    estimator=base_model_dt,
    param_grid=param_grid_dt,
    cv=cv_dt,
    scoring='accuracy',  # Optimize for accuracy
    n_jobs=-1,  # Use all available CPUs
    verbose=1,  # Show progress
    return_train_score=True  # Return training scores for overfitting check
)

# Fit grid search
import time
start_time_dt = time.time()
grid_search_dt.fit(X_train, y_train_values)
elapsed_time_dt = time.time() - start_time_dt

print(f"\n✅ Grid search completed!")
print(f"   Total time: {elapsed_time_dt/60:.1f} minutes ({elapsed_time_dt:.1f} seconds)")
print(f"   Tested {total_combinations_dt:,} parameter combinations")

# ============================================================================
# Display Best Parameters
# ============================================================================
print("\n" + "=" * 80)
print("BEST PARAMETERS FROM GRID SEARCH")
print("=" * 80)
print(f"\n🏆 Best Parameters:")
for param, value in grid_search_dt.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Best Cross-Validation Score (Accuracy): {grid_search_dt.best_score_:.4f}")

# Check for overfitting (compare train vs test CV scores)
if 'mean_train_score' in grid_search_dt.cv_results_:
    best_idx_dt = grid_search_dt.best_index_
    train_cv_score_dt = grid_search_dt.cv_results_['mean_train_score'][best_idx_dt]
    test_cv_score_dt = grid_search_dt.cv_results_['mean_test_score'][best_idx_dt]
    cv_gap_dt = train_cv_score_dt - test_cv_score_dt
    
    print(f"\n📊 Cross-Validation Scores:")
    print(f"   Train CV Score (Accuracy): {train_cv_score_dt:.4f}")
    print(f"   Test CV Score (Accuracy): {test_cv_score_dt:.4f}")
    print(f"   Gap: {cv_gap_dt:.4f}")
    
    if cv_gap_dt > 0.15:
        print(f"   ⚠️  WARNING: Large gap detected! Model may be overfitting.")
        print(f"   💡 Consider increasing min_samples_split or min_samples_leaf.")
    elif cv_gap_dt > 0.10:
        print(f"   ⚠️  CAUTION: Moderate gap detected.")
    else:
        print(f"   ✅ Good: Gap is acceptable.")
    
    # Show top 5 parameter combinations for reference
    print(f"\n📊 Top 5 Parameter Combinations (by CV Accuracy):")
    results_df_dt = pd.DataFrame(grid_search_dt.cv_results_)
    top_5_dt = results_df_dt.nlargest(5, 'mean_test_score')[['mean_test_score', 'mean_train_score', 'params']]
    for rank, (idx, row) in enumerate(top_5_dt.iterrows(), 1):
        print(f"   Rank {rank}: CV Accuracy = {row['mean_test_score']:.4f}, Train = {row['mean_train_score']:.4f}")
        # Format params for better readability
        params_str = str(row['params']).replace("'", "").replace("{", "").replace("}", "")
        print(f"      Params: {params_str}")

# ============================================================================
# Retrain on Full Training Set with Best Parameters
# ============================================================================
print("\n" + "=" * 80)
print("RETRAINING ON FULL TRAINING SET WITH BEST PARAMETERS")
print("=" * 80)

# Create model with best parameters
best_model_dt = DecisionTreeClassifier(
    max_depth=grid_search_dt.best_params_['max_depth'],
    min_samples_split=grid_search_dt.best_params_['min_samples_split'],
    min_samples_leaf=grid_search_dt.best_params_['min_samples_leaf'],
    max_features=grid_search_dt.best_params_['max_features'],
    criterion=grid_search_dt.best_params_['criterion'],
    min_impurity_decrease=grid_search_dt.best_params_['min_impurity_decrease'],
    class_weight=grid_search_dt.best_params_['class_weight'],
    random_state=42
)

# Retrain on FULL training set
print("\n🔄 Training on full training set...")
best_model_dt.fit(X_train, y_train_values)
print("✅ Model trained on full training set!")

# ============================================================================
# Evaluate on Test Set
# ============================================================================
print("\n" + "=" * 80)
print("EVALUATING ON TEST SET")
print("=" * 80)

# Make predictions
y_train_pred_dt = best_model_dt.predict(X_train)
y_test_pred_dt = best_model_dt.predict(X_test)

# Get prediction probabilities (for ROC-AUC)
y_train_proba_dt = best_model_dt.predict_proba(X_train)[:, 1]
y_test_proba_dt = best_model_dt.predict_proba(X_test)[:, 1]

# Calculate metrics
train_accuracy_dt = accuracy_score(y_train_values, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test_values, y_test_pred_dt)

train_precision_dt = precision_score(y_train_values, y_train_pred_dt, average='weighted', zero_division=0)
test_precision_dt = precision_score(y_test_values, y_test_pred_dt, average='weighted', zero_division=0)

train_recall_dt = recall_score(y_train_values, y_train_pred_dt, average='weighted', zero_division=0)
test_recall_dt = recall_score(y_test_values, y_test_pred_dt, average='weighted', zero_division=0)

train_f1_dt = f1_score(y_train_values, y_train_pred_dt, average='weighted', zero_division=0)
test_f1_dt = f1_score(y_test_values, y_test_pred_dt, average='weighted', zero_division=0)

train_auc_dt = roc_auc_score(y_train_values, y_train_proba_dt)
test_auc_dt = roc_auc_score(y_test_values, y_test_proba_dt)

# Confusion matrices
train_cm_dt = confusion_matrix(y_train_values, y_train_pred_dt)
test_cm_dt = confusion_matrix(y_test_values, y_test_pred_dt)

# Per-class metrics
train_precision_per_class_dt, train_recall_per_class_dt, train_f1_per_class_dt, _ = precision_recall_fscore_support(
    y_train_values, y_train_pred_dt, average=None, zero_division=0
)
test_precision_per_class_dt, test_recall_per_class_dt, test_f1_per_class_dt, _ = precision_recall_fscore_support(
    y_test_values, y_test_pred_dt, average=None, zero_division=0
)

# ============================================================================
# Display Results
# ============================================================================
print("\n📊 Training Set Performance:")
print(f"   Accuracy:  {train_accuracy_dt:.4f}")
print(f"   Precision: {train_precision_dt:.4f}")
print(f"   Recall:    {train_recall_dt:.4f}")
print(f"   F1 Score:  {train_f1_dt:.4f}")
print(f"   AUC-ROC:   {train_auc_dt:.4f}")

print("\n📊 Training Set Performance (Per-Class):")
print(f"   Class 0 (Not Understandable):")
print(f"      Precision: {train_precision_per_class_dt[0]:.4f}")
print(f"      Recall:    {train_recall_per_class_dt[0]:.4f} ⭐")
print(f"      F1 Score:  {train_f1_per_class_dt[0]:.4f}")
print(f"   Class 1 (Understandable):")
print(f"      Precision: {train_precision_per_class_dt[1]:.4f}")
print(f"      Recall:    {train_recall_per_class_dt[1]:.4f}")
print(f"      F1 Score:  {train_f1_per_class_dt[1]:.4f}")

print("\n📊 Test Set Performance (FINAL EVALUATION):")
print(f"   ✅ Test Accuracy:  {test_accuracy_dt:.4f}")
print(f"   ✅ Test Precision: {test_precision_dt:.4f}")
print(f"   ✅ Test Recall:    {test_recall_dt:.4f}")
print(f"   ✅ Test F1 Score:  {test_f1_dt:.4f}")
print(f"   ✅ Test AUC-ROC:    {test_auc_dt:.4f}")

print("\n📊 Test Set Performance (Per-Class):")
print(f"   Class 0 (Not Understandable):")
print(f"      Precision: {test_precision_per_class_dt[0]:.4f}")
print(f"      Recall:    {test_recall_per_class_dt[0]:.4f} ⭐ (TARGET: Improve this!)")
print(f"      F1 Score:  {test_f1_per_class_dt[0]:.4f}")
print(f"   Class 1 (Understandable):")
print(f"      Precision: {test_precision_per_class_dt[1]:.4f}")
print(f"      Recall:    {test_recall_per_class_dt[1]:.4f}")
print(f"      F1 Score:  {test_f1_per_class_dt[1]:.4f}")

print("\n📊 Confusion Matrix (Test Set):")
print("   (Rows = Actual, Columns = Predicted)")
print(f"   [[Class 0→0  Class 0→1]")
print(f"    [Class 1→0  Class 1→1]]")
print(test_cm_dt)

print("\n📊 Classification Report (Test Set):")
print(classification_report(y_test_values, y_test_pred_dt, target_names=['Not Understandable', 'Understandable']))

# ============================================================================
# Threshold Adjustment with Balanced Optimization
# ============================================================================
print("\n" + "=" * 80)
print("THRESHOLD ADJUSTMENT (BALANCED OPTIMIZATION)")
print("=" * 80)
print("\n💡 Strategy: Find threshold that improves Class 0 recall WITHOUT sacrificing overall performance")

# Store baseline metrics (threshold = 0.5)
baseline_metrics_dt = {
    'threshold': 0.5,
    'accuracy': test_accuracy_dt,
    'f1_weighted': test_f1_dt,
    'class0_recall': test_recall_per_class_dt[0],
    'class1_recall': test_recall_per_class_dt[1],
    'class0_precision': test_precision_per_class_dt[0],
    'class1_precision': test_precision_per_class_dt[1],
    'predictions': y_test_pred_dt.copy()
}

# Try different thresholds
thresholds_to_try_dt = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
threshold_results_dt = []

print("\n📊 Testing Different Thresholds:")
print(f"{'Threshold':<12} {'F1-Weighted':<14} {'Accuracy':<12} {'Class 0 Recall':<16} {'Class 1 Recall':<16} {'Status':<15}")
print("-" * 100)

for threshold in thresholds_to_try_dt:
    y_test_pred_thresh_dt = (y_test_proba_dt >= threshold).astype(int)
    
    # Calculate metrics
    acc_thresh_dt = accuracy_score(y_test_values, y_test_pred_thresh_dt)
    prec_thresh_dt, rec_thresh_dt, f1_thresh_dt, _ = precision_recall_fscore_support(
        y_test_values, y_test_pred_thresh_dt, average=None, zero_division=0
    )
    f1_weighted_thresh_dt = f1_score(y_test_values, y_test_pred_thresh_dt, average='weighted', zero_division=0)
    
    # Check if this threshold meets minimum requirements
    meets_requirements_dt = (
        rec_thresh_dt[0] >= 0.3 and  # Class 0 recall >= 0.3
        acc_thresh_dt >= 0.65 and   # Accuracy >= 0.65
        rec_thresh_dt[1] >= 0.5     # Class 1 recall >= 0.5
    )
    
    status_dt = "✅ Valid" if meets_requirements_dt else "❌ Rejected"
    
    threshold_results_dt.append({
        'threshold': threshold,
        'accuracy': acc_thresh_dt,
        'f1_weighted': f1_weighted_thresh_dt,
        'class0_recall': rec_thresh_dt[0],
        'class1_recall': rec_thresh_dt[1],
        'class0_precision': prec_thresh_dt[0],
        'class1_precision': prec_thresh_dt[1],
        'meets_requirements': meets_requirements_dt,
        'predictions': y_test_pred_thresh_dt
    })
    
    print(f"{threshold:<12.2f} {f1_weighted_thresh_dt:<14.4f} {acc_thresh_dt:<12.4f} {rec_thresh_dt[0]:<16.4f} {rec_thresh_dt[1]:<16.4f} {status_dt:<15}")

# Find best threshold: maximize F1-weighted among valid thresholds
valid_results_dt = [r for r in threshold_results_dt if r['meets_requirements']]

if valid_results_dt:
    # Sort by F1-weighted score (descending)
    valid_results_dt.sort(key=lambda x: x['f1_weighted'], reverse=True)
    best_result_dt = valid_results_dt[0]
    best_threshold_dt = best_result_dt['threshold']
    
    # Compare with baseline
    baseline_f1_dt = baseline_metrics_dt['f1_weighted']
    baseline_class0_recall_dt = baseline_metrics_dt['class0_recall']
    
    improvement_f1_dt = best_result_dt['f1_weighted'] - baseline_f1_dt
    improvement_class0_recall_dt = best_result_dt['class0_recall'] - baseline_class0_recall_dt
    
    print(f"\n✅ Best threshold found: {best_threshold_dt:.2f}")
    print(f"   F1-weighted: {baseline_f1_dt:.4f} → {best_result_dt['f1_weighted']:.4f} ({improvement_f1_dt:+.4f})")
    print(f"   Class 0 Recall: {baseline_class0_recall_dt:.4f} → {best_result_dt['class0_recall']:.4f} ({improvement_class0_recall_dt:+.4f})")
    print(f"   Accuracy: {baseline_metrics_dt['accuracy']:.4f} → {best_result_dt['accuracy']:.4f} ({best_result_dt['accuracy'] - baseline_metrics_dt['accuracy']:+.4f})")
    
    # Only use new threshold if it provides meaningful improvement
    if improvement_f1_dt > 0.01 or (improvement_class0_recall_dt > 0.1 and best_result_dt['f1_weighted'] >= baseline_f1_dt * 0.95):
        print(f"\n💡 Using optimized threshold: {best_threshold_dt:.2f}")
        y_test_pred_dt = best_result_dt['predictions']
        
        # Update metrics
        test_accuracy_dt = best_result_dt['accuracy']
        test_precision_per_class_dt = [best_result_dt['class0_precision'], best_result_dt['class1_precision']]
        test_recall_per_class_dt = [best_result_dt['class0_recall'], best_result_dt['class1_recall']]
        test_f1_dt = best_result_dt['f1_weighted']
        test_cm_dt = confusion_matrix(y_test_values, y_test_pred_dt)
        
        print(f"\n📊 Updated Test Set Performance (with threshold={best_threshold_dt}):")
        print(f"   ✅ Test Accuracy:  {test_accuracy_dt:.4f}")
        print(f"   ✅ Test F1 Score:  {test_f1_dt:.4f}")
        print(f"   ✅ Class 0 Recall: {test_recall_per_class_dt[0]:.4f}")
        print(f"   ✅ Class 1 Recall: {test_recall_per_class_dt[1]:.4f}")
        print(f"\n📊 Updated Confusion Matrix (Test Set):")
        print(test_cm_dt)
    else:
        print(f"\n💡 Keeping default threshold (0.5) - optimized threshold doesn't provide meaningful improvement")
else:
    print(f"\n⚠️  No threshold found that meets all requirements")
    print(f"💡 Keeping default threshold (0.5)")

# ============================================================================
# Overfitting Check
# ============================================================================
overfitting_gap_dt = train_accuracy_dt - test_accuracy_dt
print(f"\n⚠️  Overfitting Check:")
print(f"   Train-Test Accuracy Gap: {overfitting_gap_dt:.4f}")
if overfitting_gap_dt > 0.15:
    print(f"   ⚠️  WARNING: Large gap detected! Model may be overfitting.")
    print(f"   💡 Consider increasing min_samples_split or min_samples_leaf.")
    print(f"   💡 Current max_depth: {grid_search_dt.best_params_['max_depth']}")
    print(f"   💡 Current min_samples_split: {grid_search_dt.best_params_['min_samples_split']}")
    print(f"   💡 Current min_samples_leaf: {grid_search_dt.best_params_['min_samples_leaf']}")
    print(f"   💡 Current min_impurity_decrease: {grid_search_dt.best_params_['min_impurity_decrease']}")
elif overfitting_gap_dt > 0.10:
    print(f"   ⚠️  CAUTION: Moderate gap detected.")
else:
    print(f"   ✅ Good: Gap is acceptable.")

# ============================================================================
# Feature Importance Analysis
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# Get feature importance
feature_importance_dt = best_model_dt.feature_importances_

# Get feature names (assuming x_train is a DataFrame with column names)
if hasattr(x_train, 'columns'):
    feature_names_dt = x_train.columns.tolist()
else:
    feature_names_dt = [f'Feature_{i}' for i in range(len(feature_importance_dt))]

# Create DataFrame for easier viewing
importance_df_dt = pd.DataFrame({
    'Feature': feature_names_dt,
    'Importance': feature_importance_dt
}).sort_values('Importance', ascending=False)

print(f"\n📊 Top 10 Most Important Features:")
print(importance_df_dt.head(10).to_string(index=False))

print("\n" + "=" * 80)
print("✅ DECISION TREE TRAINING COMPLETE!")
print("=" * 80)


DECISION TREE: HYPERPARAMETER TUNING WITH OVERFITTING PREVENTION

📊 Data Preparation:
   Training set: 505 samples, 31 features
   Test set: 48 samples, 31 features
   Training labels distribution:
      - Class 0: 127 (25.1%)
      - Class 1: 378 (74.9%)

🔍 Hyperparameter Distribution (RandomizedSearchCV):
   Max depth: 15 options
   Min samples split: 14 options
   Min samples leaf: 12 options
   Max features: 7 options
   Criterion: 2 options
   Min impurity decrease: 8 options
   Class weights: 10 options
   Total possible combinations: 2,822,400

💡 Using RandomizedSearchCV to efficiently search this large space
   Will test 200 random combinations
   With 5-fold CV: ~1000 model fits

STARTING RANDOMIZED SEARCH WITH CROSS-VALIDATION

💡 Strategy: RandomizedSearchCV allows us to explore a much larger
   parameter space efficiently by testing random combinations.

⏳ This may take a while. Please wait...

Fitting 5 folds for each of 200 candidates, totalling 1000 fits

✅ Randomized sea